# Simulated two-layer demand data
Start here. Read README.md for data-generating assumptions and estimation limitations.

In [ ]:
from pathlib import Path
import pandas as pd
from simulate_data import simulate

tables, parameters = simulate()
products = tables["products"]
resellers = tables["resellers"]
panel = tables["reseller_panel"]
blp = tables["blp_product_market"]
capacities = tables["reseller_capacity"]
display(pd.DataFrame({"rows": {k: len(v) for k, v in tables.items()}}))
display(panel.head())

## Layer 1 estimation inputs
Market size includes the outside option. Use requested orders because backorders remain available.

In [ ]:
display(blp.head())
display(blp.groupby("scarcity_state").agg(rows=("sku","size"), mean_price=("price","mean"), mean_share=("share","mean")))
print("Zero inside-share rows:", blp["share"].eq(0).sum())
print("Outside-share range:", blp["outside_share"].min(), blp["outside_share"].max())

## Layer 2 estimation inputs
Reporting covers the full panel. Missing sales are not zero sales.

In [ ]:
enduser_inputs = panel.merge(resellers, on=["reseller_id", "region"], validate="many_to_one")
enduser_inputs["quarter_index"] = pd.PeriodIndex(enduser_inputs["quarter"], freq="Q").asi8 - pd.PeriodIndex(enduser_inputs["quarter"], freq="Q").asi8.min()
display(enduser_inputs.groupby("reporting_indicator").agg(rows=("sku","size"), observed_sales=("sellthrough_quantity","count")))
print("Observed zero sales:", panel["sellthrough_quantity"].eq(0).sum())
reporters = panel.groupby("reseller_id")["reporting_indicator"].sum()
print("Resellers with no reporting observations:", reporters.eq(0).sum())

## Ground truth: evaluation only
Never merge these columns into the estimator. The predicted mean is distinct from a random realized quantity.

In [ ]:
truth = tables["ground_truth"]
display(truth.head())
print(parameters)

## Save or regenerate
The downloaded data was generated with the default seed. Change the seed in simulate(seed=...) for a new experiment.

In [ ]:
output = Path("data")
output.mkdir(exist_ok=True)
for name, frame in tables.items():
    frame.to_csv(output / f"{name}.csv", index=False)